# 📊 Análise de Churn Rate: Estratégias de Retenção Baseadas em Dados.

![Python](https://img.shields.io/badge/Python-3.13.7-blue?style=for-the-badge&logo=python&logoColor=white)
![Pandas](https://img.shields.io/badge/Pandas-Data_Analysis-150458?style=for-the-badge&logo=pandas&logoColor=white)
![Status](https://img.shields.io/badge/Status-Concluído-success?style=for-the-badge)

## 1. Contexto do Negócio
Uma empresa do setor de serviços, detentora de uma base com mais de **50.000 clientes**, identificou uma erosão crítica em sua base ativa. O cenário atual aponta para uma taxa de inatividade (Churn) elevada, impactando diretamente a **Receita Recorrente Mensal (MRR)** e reduzindo o *LTV (Lifetime Value)* da companhia.

## 2. O Desafio
O objetivo central deste projeto transcende a métrica descritiva de "quantos saíram". O foco é realizar uma **Análise de Causa Raiz (Root Cause Analysis)** para entender:
* Por que os clientes estão cancelando?
* Quais perfis de clientes têm maior probabilidade de evasão?
* Quais alavancas de negócio podem ser acionadas para reverter esse cenário?

## 3. Objetivos do Projeto
* **Diagnóstico:** Analisar o comportamento histórico e transacional dos clientes.
* **Padrões de Evasão:** Identificar correlações ocultas que antecedem o cancelamento.
* **Plano de Ação:** Propor estratégias para aumentar a retenção e recuperar receita.

## 4. Configuração do Ambiente e Ingestão de Dados

Nesta etapa, inicializei o ambiente de desenvolvimento importando bibliotecas de alta performance para manipulação de dados e visualização interativa.

* **Pandas:** Utilizado para processos de ETL (Extração, Transformação e Carregamento) e manipulação tabular.
* **Plotly Express:** Escolhido para a criação de dashboards interativos, facilitando a identificação visual de outliers e tendências por parte dos stakeholders.

Abaixo, realizei o carregamento do dataset e a verificação preliminar da integridade dos dados.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

# Carregamento da base
# Mantive o CSV no mesmo diretório pra facilitar os testes locais
df_churn = pd.read_csv("cancelamentos.csv")

# --- Checagens iniciais ---

# Visão geral do tamanho da base
print(f"Volume de Dados: {df_churn.shape[0]} linhas e {df_churn.shape[1]} colunas")

# Conferência de tipos e nulos
print("\n--- Estrutura dos Dados e Tipagem ---")
df_churn.info()

# Validação
display(df_churn.head())


Volume de Dados: 50000 linhas e 12 colunas

--- Estrutura dos Dados e Tipagem ---
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   CustomerID              50000 non-null  float64
 1   idade                   50000 non-null  float64
 2   sexo                    49997 non-null  str    
 3   tempo_como_cliente      49998 non-null  float64
 4   frequencia_uso          50000 non-null  float64
 5   ligacoes_callcenter     50000 non-null  float64
 6   dias_atraso             50000 non-null  float64
 7   assinatura              50000 non-null  str    
 8   duracao_contrato        50000 non-null  str    
 9   total_gasto             50000 non-null  float64
 10  meses_ultima_interacao  50000 non-null  float64
 11  cancelou                50000 non-null  float64
dtypes: float64(9), str(3)
memory usage: 4.6 MB


,CustomerID,idade,sexo,tempo_como_cliente,frequencia_uso,ligacoes_callcenter,dias_atraso,assinatura,duracao_contrato,total_gasto,meses_ultima_interacao,cancelou
0,349936.0,23.0,Male,13.0,22.0,2.0,1.0,Standard,Annual,909.58,23.0,0.0
1,100634.0,49.0,Male,55.0,16.0,3.0,6.0,Premium,Monthly,207.00,29.0,1.0
2,301263.0,30.0,Male,7.0,1.0,0.0,8.0,Basic,Annual,768.78,7.0,0.0
3,119358.0,26.0,Male,40.0,5.0,3.0,8.0,Premium,Annual,398.00,12.0,1.0
4,130955.0,27.0,Female,17.0,30.0,5.0,6.0,Basic,Annual,507.00,15.0,1.0


## 5. Tratamento de Dados

Antes da análise, realizei uma limpeza básica para reduzir ruído e evitar problemas nas próximas etapas.

### O que foi feito

- **Remoção da coluna `CustomerID`**  
  Essa coluna é apenas um identificador e não traz informação útil para prever churn neste dataset.

- **Valores nulos**  
  Removi as linhas com valores faltantes, já que eram poucos registros e não justificavam um tratamento mais complexo.

- **Registros duplicado**
    Eliminei duplicatas para garantir que cada cliente fosse contado apenas uma vez.


In [ ]:

display(df_churn.info())


df_churn=df_churn.drop(columns="CustomerID") # Remover featura não informativa
df_churn=df_churn.dropna() # Remover linhas com valores nulos (optei por ter apenas 4 linhas com valores nulos)
df_churn=df_churn.drop_duplicates() # Remover linhas duplicadas
df_churn=df_churn.reset_index(drop=True) # Resetar índices após remoção de linhas
print("\n--- Status do Dataset Limpo ---")
display(df_churn.info())

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   CustomerID              50000 non-null  float64
 1   idade                   50000 non-null  float64
 2   sexo                    49997 non-null  str    
 3   tempo_como_cliente      49998 non-null  float64
 4   frequencia_uso          50000 non-null  float64
 5   ligacoes_callcenter     50000 non-null  float64
 6   dias_atraso             50000 non-null  float64
 7   assinatura              50000 non-null  str    
 8   duracao_contrato        50000 non-null  str    
 9   total_gasto             50000 non-null  float64
 10  meses_ultima_interacao  50000 non-null  float64
 11  cancelou                50000 non-null  float64
dtypes: float64(9), str(3)
memory usage: 4.6 MB


None


--- Status do Dataset Limpo ---
<class 'pandas.DataFrame'>
RangeIndex: 48527 entries, 0 to 48526
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   idade                   48527 non-null  float64
 1   sexo                    48527 non-null  str    
 2   tempo_como_cliente      48527 non-null  float64
 3   frequencia_uso          48527 non-null  float64
 4   ligacoes_callcenter     48527 non-null  float64
 5   dias_atraso             48527 non-null  float64
 6   assinatura              48527 non-null  str    
 7   duracao_contrato        48527 non-null  str    
 8   total_gasto             48527 non-null  float64
 9   meses_ultima_interacao  48527 non-null  float64
 10  cancelou                48527 non-null  float64
dtypes: float64(8), str(3)
memory usage: 4.1 MB


None

## 6. Diagnóstico Inicial: Análise da Variável Alvo

Antes de investigar as causas, precisamos quantificar a proporção de clientes que já deixaram a empresa em relação à base ativa.

`Qual é o Churn Rate?`

> Obs: esta análise considera apenas o status atual do cliente, sem recorte temporal.


In [ ]:
# Taxa de Churn na Base de Dados
churn_resumo = pd.DataFrame({
    "Total": df_churn["cancelou"].value_counts(),
    "Porcentagem": df_churn["cancelou"].value_counts(normalize=True).map("{:.2%}".format)
})

print("--- Tabela de Conferência: Status da Base ---")
display(churn_resumo)

# Plotar gráfico de barras para visualizar a distribuição de cancelamentos
grafico=px.histogram(df_churn, x="cancelou", color="cancelou",text_auto=True, title=f"Distribuição de Cancelamentos")
grafico.show()

--- Tabela de Conferência: Status da Base ---


,Total,Porcentagem
cancelou,,
1.0,27539,56.75%
0.0,20988,43.25%


## 7. Análise Exploratória das Variáveis

Nesta etapa, comparei as principais variáveis com o status de cancelamento (`cancelou`) para entender quais fatores parecem estar mais associados ao churn nesta base.

### Perguntas que guiaram a análise
- Clientes com contrato mensal apresentam maior taxa de cancelamento?
- Idade e tempo de relacionamento mostram alguma correlação com churn?
- Há indícios de impacto financeiro, como preço ou gasto mensal, na decisão de cancelamento?


In [ ]:


for coluna in df_churn.columns:
    if coluna != "cancelou":
        print(f"Analisando a coluna: {coluna}")

        grafico=px.histogram(df_churn, x=coluna, color="cancelou", barmode="group", title=f"Distribuição de {coluna} por Cancelamento")
        grafico.show()
        print("-" * 50)  # Linha separadora visual



Analisando a coluna: idade


--------------------------------------------------
Analisando a coluna: sexo


--------------------------------------------------
Analisando a coluna: tempo_como_cliente


--------------------------------------------------
Analisando a coluna: frequencia_uso


--------------------------------------------------
Analisando a coluna: ligacoes_callcenter


...

--------------------------------------------------
Analisando a coluna: assinatura


--------------------------------------------------
Analisando a coluna: duracao_contrato


--------------------------------------------------
Analisando a coluna: total_gasto


--------------------------------------------------
Analisando a coluna: meses_ultima_interacao


--------------------------------------------------


## 8. Insights e Possíveis Ações

A análise exploratória revelou alguns padrões fortes de comportamento associados
ao cancelamento. Abaixo estão os principais insights observados nesta base, junto
com possíveis ações de negócio que poderiam mitigar o churn.

### Principais Insights

- **Clientes com mais de 50 anos** apresentaram taxa de cancelamento muito elevada.
- **Clientes do sexo feminino** possuem maior taxa de churn em relação aos demais.
- **Contratos mensais** concentram praticamente todos os cancelamentos.
- **Baixa frequência de uso** está associada a maior propensão ao cancelamento.
- **Clientes que entram em contato com o call center mais de 5 vezes** acabam cancelando.
- **Atrasos superiores a 20 dias** estão fortemente associados ao churn.
- **Clientes com mais de 15 meses sem interação** apresentam alta taxa de cancelamento.

### Possíveis Ações de Mitigação

- **Clientes acima de 50 anos**  
  Reavaliar a experiência da plataforma (UX), considerando limitações e expectativas
  desse público.

- **Público feminino**  
  Realizar pesquisas de satisfação para entender pontos específicos de frustração
  ou oportunidade de melhoria.

- **Contratos mensais**  
  Reavaliar a oferta desse plano, incentivando a migração para contratos mais longos,
  que demonstram maior retenção.

- **Baixa frequência de uso**  
  Monitorar queda de uso e ativar comunicações proativas para estimular o engajamento.

- **Call center (mais de 5 ligações)**  
  Criar alertas para identificar clientes recorrentes e direcioná-los para um
  atendimento mais especializado.

- **Atrasos de pagamento**  
  Facilitar processos de renegociação antes que o atraso atinja um ponto crítico.

- **Longos períodos sem interação**  
  Automatizar campanhas de reengajamento com comunicações periódicas e benefícios.

### Priorização

Idade, volume de ligações ao call center e contrato mensal se destacam como os
ofensores mais relevantes, pois impactam uma parcela significativa da base e estão
diretamente ligados à experiência do cliente. Recomenda-se priorizar ações nesses
três pontos.

# Panorama após a tomada de ações

A seguir, simulamos um **cenário hipotético e otimista**, no qual os principais
ofensores identificados foram completamente mitigados. Este exercício não
representa uma previsão real, mas uma estimativa do impacto máximo que essas ações
poderiam ter na taxa de churn.

### Análise de Impacto por Ação (Cenários Isolados)

Nesta seção, avaliamos o impacto potencial de cada ação de forma isolada,
comparando a taxa de churn original com um cenário hipotético em que apenas
um ofensor é mitigado por vez.

Este exercício **não estabelece causalidade**, mas permite estimar a parcela
do churn associada a cada fator dentro desta base.


In [ ]:
# Taxa de Churn Original
churn_original = df_churn["cancelou"].value_counts(normalize=True).get(1.0, 0.0)
print(f"Churn Original: {churn_original:.2%}")

# Função para simular alterações e calcular novo churn
def simular_acao(df, filtro, alteracoes):
    df_sim = df.copy()
    idx = df_sim[filtro(df_sim)].index
    for coluna, valor in alteracoes.items():
        df_sim.loc[idx, coluna] = valor
    churn = df_sim["cancelou"].value_counts(normalize=True).get(1.0, 0.0)
    return churn

# Ação 1: Clientes acima de 50 anos
churn_idade = simular_acao(
    df_churn,
    filtro=lambda d: (d["idade"] > 50) & (d["cancelou"] == 1.0),
    alteracoes={"cancelou": 0.0}
)

# Ação 2: Call center (>4 ligações)
churn_callcenter = simular_acao(
    df_churn,
    filtro=lambda d: d["ligacoes_callcenter"] > 4,
    alteracoes={"cancelou": 0.0, "ligacoes_callcenter": 4}
)

# Ação 3: Contrato mensal
churn_contrato = simular_acao(
    df_churn,
    filtro=lambda d: d["duracao_contrato"] == "Monthly",
    alteracoes={"cancelou": 0.0, "duracao_contrato": "Annual"}
)

print("Impacto estimado por ação (cenários isolados):")
print(f"Idade 50+: {- (churn_idade - churn_original):.2%}")
print(f"Call Center: {- (churn_callcenter - churn_original):.2%}")
print(f"Contrato Mensal: {- (churn_contrato - churn_original):.2%}")


Churn Original: 56.75%
Impacto estimado por ação (cenários isolados):
Idade 50+: 18.31%
Call Center: 32.17%
Contrato Mensal: 19.77%


> Observação: os impactos não são aditivos, pois há sobreposição entre os grupos analisados.


In [ ]:
# --- Cenário Combinado (Impacto Total) ---

df_cenario_combinado = df_churn.copy()

# Ação 1: clientes acima de 50 anos
filtro_idade = df_cenario_combinado["idade"] > 50
df_cenario_combinado.loc[filtro_idade, "cancelou"] = 0.0

# Ação 2: clientes com muitas ligações ao call center
filtro_ligacoes = df_cenario_combinado["ligacoes_callcenter"] > 4
df_cenario_combinado.loc[filtro_ligacoes, "ligacoes_callcenter"] = 4
df_cenario_combinado.loc[filtro_ligacoes, "cancelou"] = 0.0

# Ação 3: contrato mensal
filtro_contrato = df_cenario_combinado["duracao_contrato"] == "Monthly"
df_cenario_combinado.loc[filtro_contrato, "duracao_contrato"] = "Annual"
df_cenario_combinado.loc[filtro_contrato, "cancelou"] = 0.0

# Churn no cenário combinado
churn_combinado = df_cenario_combinado["cancelou"].value_counts(normalize=True).get(1.0, 0.0)

print(f"Churn Original: {churn_original:.2%}")
print(f"Churn no Cenário Combinado: {churn_combinado:.2%}")
print(f"Redução Potencial Total: {(churn_original - churn_combinado):.2%}")


Churn Original: 56.75%
Churn no Cenário Combinado: 9.89%
Redução Potencial Total: 46.86%


### 💡 Análise da Sobreposição

Nota-se que a soma das reduções isoladas (~70%) supera a redução total combinada (~46%). Isso ocorre devido à **intersecção dos grupos de risco**.

Muitos clientes acumulam múltiplos fatores de risco simultaneamente (ex: um cliente >50 anos que *também* possui contrato mensal e *também* liga excessivamente ao suporte). No cenário combinado, evitamos o cancelamento desse cliente apenas uma vez, o que corrige a dupla contagem e reflete o cenário realista.

## 9. Conclusão

Este estudo teve como objetivo identificar padrões comportamentais associados ao
cancelamento de clientes e estimar, de forma exploratória, o impacto potencial de
ações de retenção baseadas nesses padrões.

A análise revelou que o churn não está distribuído de forma aleatória, mas fortemente
concentrado em alguns perfis específicos, com destaque para:
- clientes com mais de 50 anos,
- clientes com alto volume de contatos com o call center,
- clientes em contratos mensais.

Ao simular cenários isolados, foi possível estimar o **potencial máximo de impacto**
de cada ação individualmente, permitindo uma visão clara de **priorização** e
**alavancagem de negócio**. Esses cenários não representam causalidade direta,
mas sim a ordem de grandeza do problema associado a cada fator.

No cenário combinado, ao aplicar simultaneamente todas as ações simuladas,
observou-se uma redução expressiva da taxa de churn.  
⚠️ **É importante destacar que esse resultado representa um limite superior teórico**,
assumindo eficácia total das iniciativas. Na prática, parte do churn é estrutural
(ex: mudança de necessidade do cliente, fatores externos ou decisões pessoais)
e não seria eliminada apenas com intervenções operacionais.

Ainda assim, o exercício é valioso por dois motivos principais:
1. Evidencia que a maior parte do churn está concentrada em poucos fatores-chave.
2. Fornece uma base analítica sólida para tomada de decisão e desenho de
   experimentos reais (A/B tests, pilotos de retenção e ações segmentadas).

Como próximos passos, recomenda-se:
- validar essas hipóteses com testes controlados,
- incorporar variáveis adicionais (ex: NPS, renda, concorrência),
- evoluir o modelo para abordagens preditivas, estimando churn residual e
  probabilidade individual de cancelamento.

Em resumo, a análise demonstra como dados podem ser usados não apenas para
explicar o passado, mas para **orientar decisões estratégicas de retenção com foco
em impacto real de negócio**.
